# Lab F4. Density and probability forecasts

A point forecast says where; a density forecast also says how sure. This lab scores densities, checks whether
they are calibrated, and turns them into probabilities of an event ("will growth be negative?") that can be
scored in their turn.

**Help:** for any pymetrix function, type its name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call. The last cell lists every pymetrix function, class and method this lab uses.

Topics: the log score and the CRPS; the probability integral transform (PIT) and the Berkowitz test; interval
forecasts and the Christoffersen test; rolling and expanding windows; probability forecasts of events: the Brier
score, the ROC curve, the Kuipers score and the Pesaran-Timmermann test (Elliott and Timmermann, chapters 12,
13, 16 and 18).

The data: 240 quarters of growth, $y_t = 0.5 + 0.5\,y_{t-1} + \sigma_t \varepsilon_t$, where the volatility
$\sigma_t$ doubles from 1 to 2 in 2008Q1 and stays there.

In [ ]:
import numpy as np
import pandas as pd
from scipy import integrate, stats

from pymetrix import Dataset, Equation, backtest
from pymetrix.stats import binary_scores, brier_score, pesaran_timmermann, roc_curve

rng = np.random.default_rng(16)
index = pd.period_range("1965Q1", periods=240, freq="Q")
sigma = np.where(index >= pd.Period("2008Q1", freq="Q"), 2.0, 1.0)
y = np.zeros(len(index))
y[0] = 1.0
for t in range(1, len(index)):
    y[t] = 0.5 + 0.5 * y[t - 1] + sigma[t] * rng.standard_normal()
data = Dataset.from_frame(pd.DataFrame({"y": y}, index=index), source="simulated, volatility break in 2008Q1")

## 1. Density forecasts from two windows

Both forecasts come from the right model, an AR(1), with a normal density from its standard error. One estimates
on all the data to each origin (an expanding window); the other on the last 40 quarters only (a rolling window).
One step ahead, from 1990Q1 to 2024Q3.

In [ ]:
ar = Equation("y ~ y[-1]").fit(data)
origins = ("1990Q1", "2024Q3")
expanding = backtest(ar, data, origins, horizon=1, label="expanding")
rolling = backtest(ar, data, origins, horizon=1, window=40, label="rolling, 40 quarters")
pd.concat([expanding.density_scores(), rolling.density_scores()], keys=["expanding", "rolling"])

**Check it yourself.** For a normal density the CRPS has a closed form; it is also
$\int (F(z) - 1\{y \le z\})^2\,dz$, which can be integrated numerically. Take the first forecast.

In [ ]:
row = expanding.errors.iloc[0]
F = stats.norm(row["forecast"], row["sd"]).cdf
crps_numeric = (integrate.quad(lambda z: F(z) ** 2, -np.inf, row["actual"])[0]
                + integrate.quad(lambda z: (1 - F(z)) ** 2, row["actual"], np.inf)[0])
log_score = stats.norm(row["forecast"], row["sd"]).logpdf(row["actual"])
assert np.isclose(crps_numeric, row["crps"]) and np.isclose(log_score, row["log score"])
print(f"CRPS by integration {crps_numeric:.6f}, pymetrix {row['crps']:.6f}; log score {log_score:.6f}, pymetrix {row['log score']:.6f}")

## 2. Are the densities calibrated?

If the densities are right, the PITs, $u_t = F_t(y_t)$, are independent and uniform: outcomes land in each
tenth of the forecast distribution a tenth of the time. Berkowitz transforms them to normals and tests mean 0,
variance 1 and no autocorrelation; Christoffersen tests whether the right share of outcomes falls outside the 90%
intervals, and whether the misses come in clusters.

In [ ]:
expanding.evaluation()

In [ ]:
rolling.evaluation()

In [ ]:
expanding.plot_pit()

Split the expanding window's PITs at the break: before 2008 its densities are fine; after, they are too narrow,
and too many outcomes land in the tails.

In [ ]:
pit = expanding.pit("y")
before, after = pit[pit.index < pd.Period("2008Q1", freq="Q")], pit[pit.index >= pd.Period("2008Q1", freq="Q")]
outside = lambda u: float(np.mean((u < 0.05) | (u > 0.95)))
{"outside the 90% interval, before 2008": round(outside(before), 3), "after 2008": round(outside(after), 3)}

**Questions.**

1. The rolling window has better-calibrated densities after the break. What does it cost before the break?
   (Compare `expanding.scores()` and `rolling.scores()` on the forecasts before 2008.)
2. Compare the two density forecasts with the log score and the CRPS
   (`rolling.compare(expanding, "y", 1, loss="crps")`). Does the test agree with the calibration tests?
3. A model of the changing variance (a GARCH, **Estimate → GARCH** with the `arch` package) is the other
   answer. Why might it still be beaten by the rolling window here?

## 3. Probabilities of an event

The chance that growth is negative next quarter is the density's mass below zero, $\Phi(-\hat y/\hat\sigma)$.
Score the two sets of probabilities. The Brier score is the mean squared difference between the probability and
the outcome (1 when growth was negative); lower is better.

In [ ]:
def probabilities(bt):
    e = bt.errors.set_index("target")
    return pd.Series(stats.norm.cdf(-e["forecast"] / e["sd"]), index=e.index), (e["actual"] < 0).astype(int)


p_exp, negative = probabilities(expanding)
p_roll, _ = probabilities(rolling)
assert np.isclose(brier_score(p_exp, negative), np.mean((p_exp - negative) ** 2))
pd.DataFrame({name: binary_scores(p, negative).to_frame().set_index("")["value"] for name, p in (("expanding", p_exp), ("rolling", p_roll))})

The ROC curve plots the hit rate against the false-alarm rate as the probability threshold for "call it" falls;
the area under it (AUC) is the chance that a quarter with negative growth got a higher probability than one
without. The Kuipers score is the hit rate less the false-alarm rate at a threshold of 0.5.

The Pesaran-Timmermann test asks whether the forecasts get the direction right more often than chance, given how
often each direction occurs.

In [ ]:
roc = roc_curve(p_roll, negative)
roc.iloc[:: max(len(roc) // 10, 1)]

In [ ]:
e = rolling.errors.set_index("target")
pt = pesaran_timmermann(e["actual"], e["forecast"])
pt

**Questions.**

4. Negative quarters are rare before 2008 and common after. How does that affect the Brier score of a forecaster
   who always says 0.1? (`brier_score(np.full(len(negative), 0.1), negative)`.)
5. The AUC ignores calibration: multiply every probability by 0.5 and recompute it, and the Brier score. Which
   changes, and why?
6. The rolling window's densities are better calibrated, yet the expanding window's probabilities of negative
   growth have the lower Brier score. How can both be true? (Look at the Brier scores before and after 2008.)

In the app: a backtest's window has Density and Evaluation tabs (log score, CRPS, coverage, Berkowitz and
Christoffersen), and **Forecast → Backtest** has the rolling window on its Advanced tab.

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `backtest(fitted, data, origins, horizon, vintages, variable, label, window)` | Re-estimate at each origin from `origins[0]` to `origins[1]` and forecast `horizon` periods ahead. |
| `binary_scores(probabilities, outcomes, threshold, data)` | Probability forecasts of a 0/1 outcome at a glance: Brier score (and the base rate's), log score, ROC area, and the hit rate, false-alarm rate and Kuipers score of forecasting an event when the probability is at least `threshold`. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |

### From `pymetrix.stats`

| | |
|---|---|
| `brier_score(probabilities, outcomes)` | The Brier score: the mean squared difference between the forecast probability and the 0/1 outcome (lower is better). |
| `pesaran_timmermann(actual, forecast)` | Pesaran and Timmermann's (1992) test of directional accuracy: do the forecasts get the sign of the outcome right more often than independent guesses with the same frequencies would? Standard normal, one-sided. |
| `roc_curve(probabilities, outcomes)` | The ROC curve: for each threshold (the distinct forecasts, highest first), the false and true positive rates of forecasting an event when the probability is at or above it. |

### Methods and properties of the results

| | |
|---|---|
| `Backtest.density_scores(variables)` | By variable and horizon: n, the average log score (higher is better), the average CRPS (lower is better, in the variable's units), and the shares of outcomes inside the central 50% and 90% predictive intervals (0.5 and 0.9 when the densities are well calibrated). |
| `Backtest.evaluation(variables, coverage)` | The tests of each variable and horizon at a glance (p-values): Mincer-Zarnowitz efficiency, and for density forecasts the Berkowitz test of the PITs and the Christoffersen test of the `coverage` intervals. |
| `Backtest.pit(variable, horizon)` | The probability integral transforms of the forecasts of `variable` at `horizon`, by target period. |
| `Backtest.plot_pit(variables, horizon, bins)` | Histograms of the PITs at `horizon`, as densities (1 throughout for well-calibrated forecasts): a hump means the predictive densities are too wide, a U too narrow, a slope a biased forecast. |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |
| `Table.to_frame()` | The table as a pandas DataFrame: one column per table column, indexed by the row labels if it has them. |